# HungarianImitator — Training Notebook

Centralised O(N²) Hungarian imitator. Full unmasked cross-attention — every drone
sees every slot, every drone, globally. No comm-radius restriction.

**Use this when the decentralised models (rec_auction, super_decent_auction) cannot
reach cost ratio below ~1.020.** This model sets the upper bound for what is achievable
with global context and serves as a fair comparison baseline.

| Component | Role |
|---|---|
| `HungarianImitatorLayer` | Full self + bidirectional cross-attention (no mask) |
| `sinkhorn()` | Differentiable doubly-stochastic projection (training only) |
| `greedy_bijection_assignment` | Confidence-ordered greedy decode — bijection guaranteed |
| `hungarian_imitator_loss` | CE + regret + margin + coverage |

**Warm-start path** (recommended): loads formation embeddings, MLP projectors, and
value head from `superglue_negotiator_best.pt`, freezes them for 10 epochs while the
new full-attention layers adapt, then jointly fine-tunes everything.

**Cold-start path**: trains from scratch if no SuperGlue checkpoint exists.

Use the `torch_env` kernel.

In [16]:
import os, random
import numpy as np
import torch

from local_negotiator_v2 import (
    load_negotiator_dataset,
    prepare_dataset,
    evaluate_strict_decentralized,
    LocalNegotiatorGNN,
)
from superglue_negotiator import SuperGlueSwarmMatcher
import importlib
import hungarian_imitator_v2

importlib.reload(hungarian_imitator_v2)

from hungarian_imitator_v2 import (
    HungarianImitator,
    train_hungarian_imitator,
    evaluate_hungarian_imitator,
)

print("Reloaded hungarian_imitator_v2")

SEED              = 42
DATASET           = './dataset/negotiator_dataset_v1.pt'
SUPERGLUE_CKPT    = 'superglue_negotiator_best_v2.pt'
GOSSIP_CKPT       = 'strict_local_negotiator_best_v1.pt'
OUT               = './model/hungarian_imitator_best.pt'

EPOCHS            = 80
LR                = 3e-4
MAX_TRAIN         = 8000
MAX_VAL           = 500
MAX_TEST          = 500
EVAL_SUBSET       = 200
MIN_BIJ_FOR_BEST  = 0.90
FREEZE_EPOCHS     = 10

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}  |  Torch : {torch.__version__}')

Reloaded hungarian_imitator_v2
Device : cuda  |  Torch : 2.10.0+cu128


In [12]:
# ── Build model ───────────────────────────────────────────────────────────────
if os.path.isfile(SUPERGLUE_CKPT):
    print(f'Warm-starting from: {SUPERGLUE_CKPT}')
    sg_ckpt = torch.load(SUPERGLUE_CKPT, map_location='cpu', weights_only=False)
    matcher = SuperGlueSwarmMatcher()
    matcher.load_state_dict(sg_ckpt['model_state_dict'])
    model = HungarianImitator.from_superglue(matcher)
    freeze_projectors_epochs = FREEZE_EPOCHS
    if 'metrics' in sg_ckpt and 'test' in sg_ckpt['metrics']:
        m = sg_ckpt['metrics']['test']
        print(f'  SuperGlue bij={m.get("bijection_rate","?"):.3f}  '
              f'cost={m.get("cost_ratio_vs_hungarian","?"):.4f}  '
              f'match={m.get("slot_match_rate","?"):.3f}')
    print(f'  Projectors frozen for first {freeze_projectors_epochs} epochs')
else:
    print(f'No SuperGlue checkpoint at {SUPERGLUE_CKPT} — cold start.')
    model = HungarianImitator()
    freeze_projectors_epochs = 0

model = model.to(DEVICE)
total_p = sum(p.numel() for p in model.parameters())
attn_p  = sum(p.numel() for p in model.layers.parameters())
print(f'Total params     : {total_p:,}')
print(f'Attention params : {attn_p:,}  (always trained)')
print(f'Projector params : {total_p - attn_p:,}  (frozen {freeze_projectors_epochs} epochs)')

Warm-starting from: superglue_negotiator_best_v2.pt
  SuperGlue bij=0.464  cost=1.0306  match=0.767
  Projectors frozen for first 10 epochs
Total params     : 617,633
Attention params : 598,272  (always trained)
Projector params : 19,361  (frozen 10 epochs)


In [13]:
# ── Dataset ───────────────────────────────────────────────────────────────────
raw = load_negotiator_dataset(DATASET)
train_data, val_data, test_data = prepare_dataset(
    raw,
    model.formation_embedding.weight.detach().cpu(),
    seed=SEED,
    force_gt_visibility=False,
)
train_data = train_data[:MAX_TRAIN]
val_data   = val_data[:MAX_VAL]
test_data  = test_data[:MAX_TEST]
print(f'Train={len(train_data)}  Val={len(val_data)}  Test={len(test_data)}')

Train=8000  Val=500  Test=500


In [14]:
# ── Baselines ─────────────────────────────────────────────────────────────────
print('=== Baselines (test set, before training) ===')

gossip_final = {}
if os.path.isfile(GOSSIP_CKPT):
    gossip_base = LocalNegotiatorGNN().to(DEVICE)
    g_ckpt = torch.load(GOSSIP_CKPT, map_location='cpu', weights_only=False)
    gossip_base.load_state_dict(g_ckpt['model_state_dict'])
    gossip_base.to(DEVICE)
    print('\nGossip consensus (decentralised baseline):')
    gossip_final = evaluate_strict_decentralized(gossip_base, test_data[:200], DEVICE)
    for k, v in gossip_final.items(): print(f'  {k}: {v:.4f}')

print('\nHungarianImitator (before training):')
baseline = evaluate_hungarian_imitator(model, test_data[:200], DEVICE)
for k, v in baseline.items(): print(f'  {k}: {v:.4f}')
print(f'\nNote: centralised full-attention gives high bijection even untrained')
print('because greedy decoding on full-visibility values rarely conflicts.')

=== Baselines (test set, before training) ===

Gossip consensus (decentralised baseline):
  bijection_rate: 0.6300
  conflict_rate: 0.0221
  unassigned_rate: 0.0138
  cost_ratio_vs_hungarian: 1.0325
  slot_match_rate: 0.6508
  consensus_rounds: 9.4000
  converged_rate: 0.5400

HungarianImitator (before training):
  bijection_rate: 1.0000
  conflict_rate: 0.0000
  unassigned_rate: 0.0000
  cost_ratio_vs_hungarian: 1.2208
  slot_match_rate: 0.1896
  consensus_rounds: 0.0000

Note: centralised full-attention gives high bijection even untrained
because greedy decoding on full-visibility values rarely conflicts.


In [17]:
# ── Train ─────────────────────────────────────────────────────────────────────
print('=== Training HungarianImitator ===')
if freeze_projectors_epochs > 0:
    print(f'Phase 1 (ep 1-{freeze_projectors_epochs}): attention layers only')
    print(f'Phase 2 (ep {freeze_projectors_epochs+1}-{EPOCHS}): all params, projectors at 0.1x LR')
print()

history = train_hungarian_imitator(
    model,
    train_data,
    val_data,
    DEVICE,
    epochs                   = EPOCHS,
    lr                       = LR,
    min_bijection_for_best   = MIN_BIJ_FOR_BEST,
    eval_subset              = EVAL_SUBSET,
    ckpt_path                = OUT,
    freeze_projectors_epochs = freeze_projectors_epochs,
)

=== Training HungarianImitator ===
Phase 1 (ep 1-10): attention layers only
Phase 2 (ep 11-80): all params, projectors at 0.1x LR

epoch 001 [frozen] train=0.7361 (ce=0.625 reg=0.064 mrg=0.735) val=0.5959 bij=1.000 cost=1.0188 match=0.629
epoch 002 [frozen] train=0.5949 (ce=0.503 reg=0.056 mrg=0.631) val=0.5312 bij=1.000 cost=1.0223 match=0.628
epoch 003 [frozen] train=0.5297 (ce=0.444 reg=0.049 mrg=0.611) val=0.4453 bij=1.000 cost=1.0150 match=0.684
epoch 004 [frozen] train=0.4899 (ce=0.413 reg=0.045 mrg=0.562) val=0.4521 bij=1.000 cost=1.0172 match=0.688
epoch 005 [frozen] train=0.4595 (ce=0.390 reg=0.042 mrg=0.513) val=0.4076 bij=1.000 cost=1.0167 match=0.717
epoch 006 [frozen] train=0.4309 (ce=0.367 reg=0.040 mrg=0.470) val=0.4005 bij=1.000 cost=1.0167 match=0.727
epoch 007 [frozen] train=0.4152 (ce=0.354 reg=0.038 mrg=0.456) val=0.3677 bij=1.000 cost=1.0144 match=0.727
epoch 008 [frozen] train=0.4021 (ce=0.343 reg=0.037 mrg=0.437) val=0.4199 bij=1.000 cost=1.0129 match=0.754
epoch

KeyboardInterrupt: 

In [ ]:
# ── Final evaluation ──────────────────────────────────────────────────────────
print('=== Final evaluation (test set) ===')
final = evaluate_hungarian_imitator(model, test_data, DEVICE)

keys = ['bijection_rate', 'cost_ratio_vs_hungarian', 'slot_match_rate',
        'conflict_rate', 'unassigned_rate', 'consensus_rounds']

print(f'{"metric":<35} {"gossip":>12} {"hung_imitator":>15}')
print('-' * 65)
for k in keys:
    g = gossip_final.get(k, float('nan'))
    h = final.get(k, 0.0)
    better = ''
    if k in ('bijection_rate', 'slot_match_rate') and h > g: better = ' ↑'
    if k in ('cost_ratio_vs_hungarian', 'conflict_rate',
              'unassigned_rate', 'consensus_rounds') and h < g: better = ' ↓'
    g_str = f'{g:.4f}' if g == g else 'N/A   '
    print(f'{k:<35} {g_str:>12} {h:>15.4f}{better}')

print()
print('This model is CENTRALISED — requires global state at inference.')
print('Its cost ratio is the upper bound for the decentralised models.')

In [ ]:
# ── Save ──────────────────────────────────────────────────────────────────────
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'warmstart_checkpoint':     SUPERGLUE_CKPT if os.path.isfile(SUPERGLUE_CKPT) else None,
        'epochs':                   EPOCHS,
        'lr':                       LR,
        'freeze_projectors_epochs': freeze_projectors_epochs,
        'sinkhorn_iters':           model.sinkhorn_iters,
        'sinkhorn_temp':            model.sinkhorn_temp,
        'inference':                'centralised full cross-attention + greedy bijection decode',
    },
    'history': history,
    'metrics': {
        'hungarian_imitator': final,
        'gossip_baseline':    gossip_final,
    },
}, OUT)
print(f'Saved → {OUT}')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('HungarianImitator Training', fontsize=13, fontweight='bold')
ep = range(1, len(history['train_loss']) + 1)

ax = axes[0]
ax.plot(ep, history['train_loss'],   label='train total', color='steelblue', linewidth=1.5)
ax.plot(ep, history['val_loss'],     label='val total',   color='coral',     linewidth=1.5, linestyle='--')
ax.plot(ep, history['train_ce'],     label='train CE',    color='royalblue', linewidth=1.0, alpha=0.6)
ax.plot(ep, history['train_regret'], label='train regret',color='orange',    linewidth=1.0, alpha=0.6)
ax.plot(ep, history['train_margin'], label='train margin',color='purple',    linewidth=1.0, alpha=0.6)
if freeze_projectors_epochs > 0:
    ax.axvline(freeze_projectors_epochs, color='gray', linestyle=':', alpha=0.7, label='unfreeze')
ax.set_title('Loss'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(ep, history['val_bijection_rate'],  label='bijection',  color='green',  linewidth=1.5)
ax.plot(ep, history['val_slot_match_rate'],  label='match rate', color='purple', linewidth=1.5, linestyle='--')
ax.plot(ep, history['val_conflict_rate'],    label='conflict',   color='red',    linewidth=1.0, linestyle=':', alpha=0.7)
ax.plot(ep, history['val_unassigned_rate'],  label='unassigned', color='salmon', linewidth=1.0, linestyle=':', alpha=0.7)
ax.axhline(MIN_BIJ_FOR_BEST, color='green', linestyle=':', alpha=0.4, label=f'bij threshold')
if freeze_projectors_epochs > 0:
    ax.axvline(freeze_projectors_epochs, color='gray', linestyle=':', alpha=0.7)
ax.set_title('Assignment Quality'); ax.legend(fontsize=7); ax.grid(alpha=0.3)

ax = axes[2]
ax.plot(ep, history['val_cost_ratio_vs_hungarian'], label='cost ratio', color='orange', linewidth=1.5)
ax.axhline(1.0, color='gray', linestyle='-', alpha=0.3, label='optimal')
if freeze_projectors_epochs > 0:
    ax.axvline(freeze_projectors_epochs, color='gray', linestyle=':', alpha=0.7, label='unfreeze')
ax.set_title('Cost Ratio vs Hungarian'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('hungarian_imitator_training_curves.png', dpi=120)
plt.show()
print('Saved hungarian_imitator_training_curves.png')

In [ ]:
# ── Reload best checkpoint and verify ────────────────────────────────────────
print('=== Reloading best checkpoint ===')
best_ckpt = torch.load(OUT, map_location='cpu', weights_only=False)
print(f'Best checkpoint epoch : {best_ckpt["epoch"]}')
print(f'Val metrics at save   :')
for k, v in best_ckpt['metrics'].items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

model.load_state_dict(best_ckpt['model_state_dict'])
model.to(DEVICE)
test_best = evaluate_hungarian_imitator(model, test_data, DEVICE)
print(f'\nTest set with best checkpoint:')
for k, v in test_best.items(): print(f'  {k}: {v:.4f}')